# 04 · Rebuild accepted net-water route quantities

Replays numerical calculations from an approved route-evidence export. It does **not** choose source/proxy routes or recreate the 37-column scientific net release.

This notebook contains the scientific Python logic directly, not only a call to a .py script. Run cells in order after placing approved inputs under `data/`. All outputs are written to new directories under `runs/`; source files are never overwritten. For scientific scope and frozen evidence dependencies, see `docs/reproducibility.md`.

## Scientific methods and checks

In [ ]:
#!/usr/bin/env python3
"""Rebuild net-water numerical columns from an approved Cell-109 route layer.

This is not the source-selection model. Accepted route values, confidence and
area support must be generated/approved upstream using the research workflow.
"""
from __future__ import annotations
import argparse
from pathlib import Path
import numpy as np
import pandas as pd

REQUIRED=['target_crop','country_iso3','year','pathway','SPAM_irrigated_area_ha',
          'route_value_mm_yr','route_group','route_type','confidence','semantic_status','final_value_available']
KEY=['country_iso3','fable_product','year','pathway']


def rebuild(routes):
    missing=set(REQUIRED)-set(routes.columns)
    if missing:raise ValueError(f'Accepted route fields missing: {sorted(missing)}')
    out=routes[REQUIRED].rename(columns={'target_crop':'fable_product','route_value_mm_yr':'water_requirement_mm_yr'}).copy()
    out['country_iso3']=out.country_iso3.astype(str).str.strip().str.upper()
    out['fable_product']=out.fable_product.astype(str).str.strip().str.lower()
    out['year']=pd.to_numeric(out.year,errors='raise').astype(int)
    out['water_requirement_mm_yr']=pd.to_numeric(out.water_requirement_mm_yr,errors='raise')
    out['SPAM_irrigated_area_ha']=pd.to_numeric(out.SPAM_irrigated_area_ha,errors='raise')
    if out.duplicated(KEY).any() or out[KEY].isna().any().any():raise ValueError('Non-unique/null accepted route keys')
    if not np.isfinite(out[['water_requirement_mm_yr','SPAM_irrigated_area_ha']].to_numpy(float)).all():
        raise ValueError('Missing/non-finite accepted values')
    if (out[['water_requirement_mm_yr','SPAM_irrigated_area_ha']]<0).any().any():
        raise ValueError('Negative requirement or irrigated area')
    out['water_requirement_m3_ha_yr']=out.water_requirement_mm_yr*10.0
    out['total_requirement_m3_before_system_losses']=out.water_requirement_m3_ha_yr*out.SPAM_irrigated_area_ha
    out['total_requirement_Mm3_before_system_losses']=out.total_requirement_m3_before_system_losses/1e6
    out['direct_target_LPJmL_simulation']=out.route_group.eq('DIRECT_LPJML')
    out['uses_proxy_or_harmonisation']=~out.direct_target_LPJmL_simulation
    out['geographic_completion_used']=out.route_group.astype(str).str.startswith(('GEOGRAPHIC_COMPLETION_','RUBBER_FINAL_F2','RUBBER_FINAL_F3'))
    out['irrigation_technology_adjusted']=False
    out['system_efficiency_applied']=False
    out['water_availability_constrained']=False
    out['requirement_semantic']='FABLE_HARMONISED_CROP_IRRIGATION_REQUIREMENT_ON_LPJML_PIRRWW_COMPATIBLE_SCALE_BEFORE_EXTERNAL_IRRIGATION_SYSTEM_LOSSES'
    return out.sort_values(KEY).reset_index(drop=True)


## Inputs and output location

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'
ROUTE_FILE = DATA / 'crop' / 'accepted-routes.csv.gz'
OUTPUT_FILE = RUNS / 'net-routes-01' / 'net-routes.csv.gz'
EXPECTED_ROWS = 29338


## Execute and inspect

In [ ]:
if not ROUTE_FILE.is_file():
    raise FileNotFoundError(f'Approved route evidence is required: {ROUTE_FILE}')
if OUTPUT_FILE.exists():
    raise FileExistsError(OUTPUT_FILE)
net_routes = rebuild(pd.read_csv(ROUTE_FILE))
if len(net_routes) != EXPECTED_ROWS:
    raise ValueError(f'Expected {EXPECTED_ROWS:,} rows; got {len(net_routes):,}')
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
net_routes.to_csv(OUTPUT_FILE, index=False)
print(f'ACCEPTED_NET_NUMERIC_REBUILT: rows={len(net_routes):,} -> {OUTPUT_FILE}')
